# CrimeLLM — Fine-tune InLegalBERT (3-class)

Path A: fine-tune an encoder into a `no / yes / unclear` classifier on the **crime** axis.

**Hardware:** auto-detects CUDA (NVIDIA on Windows) or MPS (Apple Silicon) or CPU. No code change between platforms.

**Setup (run once in a terminal at project root):**

```bash
uv sync
uv run python -m ipykernel install --user --name crimellm --display-name "CrimeLLM (uv)"
```

Then pick the **CrimeLLM (uv)** kernel in Jupyter.

## 1. Environment & device check

In [1]:
import sys, platform
import torch
from crimellm import resolve_device

info = resolve_device()
print(f"Python : {sys.version.split()[0]}")
print(f"Torch  : {torch.__version__}")
print(f"OS     : {platform.system()} {platform.machine()}")
print(f"Device : {info}")

Python : 3.11.14
Torch  : 2.12.0
OS     : Darwin arm64
Device : mps:Apple Silicon (arm64) (fp16=False, bf16=False)


## 2. Load data

Either use the tiny built-in sample (smoke test) or point at your own CSV with `text,label` columns where `label ∈ {0,1,2}` (`no`, `yes`, `unclear`).

In [2]:
from pathlib import Path
from crimellm import load_dataset_from_csv, load_sample_dataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_path = PROJECT_ROOT / "data" / "sample.csv"

if csv_path.exists():
    splits = load_dataset_from_csv(csv_path)
else:
    splits = load_sample_dataset()

print(splits)
splits["train"][:3]

Casting the dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})


{'text': ['There was shouting but the cause was unknown.',
  'They argued loudly in the street late at night.',
  'They smuggled cigarettes across the border at night.'],
 'label': [2, 2, 1]}

## 3. Configure

In [3]:
from crimellm import Config

cfg = Config(
    model_name="distilbert-base-uncased",  # try "microsoft/deberta-v3-base" for general text
    max_len=256,
    num_train_epochs=4,
    learning_rate=2e-5,
    train_batch_size=8,
    eval_batch_size=8,
    output_dir=str(PROJECT_ROOT / "crime_classifier"),
)
cfg

Config(model_name='distilbert-base-uncased', max_len=256, num_train_epochs=4, learning_rate=2e-05, train_batch_size=8, eval_batch_size=8, output_dir='/Users/paolobozzini/Desktop/CrimeLLM/crime_classifier', seed=42, test_size=0.33, freeze_encoder=True, id2label={0: 'no', 1: 'yes', 2: 'unclear'})

## 4. Train

In [4]:
from crimellm import train

result = train(splits, cfg)
result.eval_metrics

[crimellm] device: mps:Apple Silicon (arm64) (fp16=False, bf16=False)


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
No log,1.092354,4,0.250000,0.133333


[crimellm] final eval: {'eval_loss': 1.0923540592193604, 'eval_accuracy': 0.25, 'eval_macro_f1': 0.13333333333333333}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.0923540592193604,
 'eval_accuracy': 0.25,
 'eval_macro_f1': 0.13333333333333333}

## 5. Inference on new text

In [ ]:
from crimellm import Classifier

clf = Classifier(cfg.output_dir, max_len=cfg.max_len)

examples = [
    "He broke into the shop and emptied the till.",
    "She returned the wallet she found on the bench.",
    "Voices were heard but nothing was visible.",
]
for t in examples:
    print(f"{clf.predict(t):>8}  | {clf.predict_proba(t)}  | {t}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

      no  | {'no': 0.33406880497932434, 'yes': 0.33319321274757385, 'unclear': 0.3327379524707794}  | He broke into the shop and emptied the till.
      no  | {'no': 0.34131258726119995, 'yes': 0.3264753222465515, 'unclear': 0.33221209049224854}  | She returned the wallet she found on the bench.
      no  | {'no': 0.3446808457374573, 'yes': 0.32542484998703003, 'unclear': 0.3298943042755127}  | Voices were heard but nothing was visible.
      no  | {'no': 0.3437853157520294, 'yes': 0.3247256875038147, 'unclear': 0.3314889669418335}  | i killed him
